# [12-8강] CNN 종합 실습 제출 - 실습

In [ ]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 문제 1. CNN 종합 모델 클래스 작성하기

Conv block과 classifier head를 가진 CNN 클래스를 직접 작성합니다.

In [2]:
class SubmitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: features와 classifier를 완성하세요.
        self.features = nn.Sequential(
            nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(),
            nn.Conv2d(4, 8, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(8 * 4 * 4, 2)
    def forward(self, x):
        h = self.features(x)
        return self.classifier(h.view(h.size(0), -1))

model = SubmitCNN()
print(model(torch.randn(3, 1, 8, 8)).shape)


torch.Size([3, 2])


### 해설 및 실행 결과 해석

- 출력이 `[3, 2]`이면 3개의 이미지에 대해 2개 class logits를 만든 것입니다. forward 흐름이 제출 가능한 CNN 구조로 연결되었습니다.

## 문제 2. 종합 학습/검증 루프 실행하기

작성한 CNN을 toy image data에 학습시키고 train/valid metric을 기록합니다.

In [8]:
def make_toy_images(n=48, size=8):
    # class 0: 세로선, class 1: 가로선
    x = torch.zeros(n, 1, size, size)
    y = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        if i % 2 == 0:
            x[i, 0, :, 3:5] = 1.0
            y[i] = 0
        else:
            x[i, 0, 3:5, :] = 1.0
            y[i] = 1
    x += 0.05 * torch.randn_like(x)
    return x, y

images, labels = make_toy_images()
train_ds = TensorDataset(images[:40], labels[:40])
valid_ds = TensorDataset(images[40:], labels[40:])
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False)

class SubmitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Linear(4 * 4 * 4, 2)
    def forward(self, x):
        h = self.features(x)
        return self.classifier(h.view(h.size(0), -1))
model = SubmitCNN().to(device)

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

# TODO: loss_fn, optimizer를 만들고 3 epoch 결과를 history에 저장하세요.
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)
history = []
for epoch in range(3):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
    valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn)
    history.append((train_loss, train_acc, valid_loss, valid_acc))
print(history)


[(0.7597801089286804, 0.275, 0.5912885665893555, 1.0), (0.4862098515033722, 1.0, 0.3265261650085449, 1.0), (0.2347997784614563, 1.0, 0.11418309062719345, 1.0)]


### 해설 및 실행 결과 해석

- history에 epoch별 결과가 쌓이면 제출 시 학습이 실제로 진행됐는지 확인할 수 있습니다. valid accuracy는 모델이 학습 데이터 밖에서도 패턴을 맞히는지 보여줍니다.`

## 문제 3. batch inference 결과 만들기

학습된 모델로 validation batch를 예측하고, 사람이 읽을 수 있는 class 이름과 confidence 리스트를 만듭니다. **굵은 텍스트**


In [ ]:
def make_toy_images(n=48, size=8):
    # class 0: 세로선, class 1: 가로선
    x = torch.zeros(n, 1, size, size)
    y = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        if i % 2 == 0:
            x[i, 0, :, 3:5] = 1.0
            y[i] = 0
        else:
            x[i, 0, 3:5, :] = 1.0
            y[i] = 1
    x += 0.05 * torch.randn_like(x)
    return x, y

images, labels = make_toy_images()
train_ds = TensorDataset(images[:40], labels[:40])
valid_ds = TensorDataset(images[40:], labels[40:])
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False)

model = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(4 * 8 * 8, 2)).to(device)
loss_fn = nn.CrossEntropyLoss(); optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

for _ in range(3):
    train_one_epoch(model, train_loader, loss_fn, optimizer)
xb, yb = next(iter(valid_loader)); xb = xb.to(device)

class_to_idx = {'vertical': 0, 'horizontal': 1}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

# TODO: eval/no_grad, softmax, argmax를 사용해 class_name과 confidence를 정리하세요.
## 이부분은 다시해보기
model.eval()
with torch.no_grad:
    logits = model(xb)
    probs = F.softmax(logits, dim=1)

prediction_results = []
print(prediction_results)


In [ ]:
#####
def make_toy_images(n=48, size=8):
    # class 0: 세로선, class 1: 가로선
    x = torch.zeros(n, 1, size, size)
    y = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        if i % 2 == 0:
            x[i, 0, :, 3:5] = 1.0
            y[i] = 0
        else:
            x[i, 0, 3:5, :] = 1.0
            y[i] = 1
    x += 0.05 * torch.randn_like(x)
    return x, y

images, labels = make_toy_images()
train_ds = TensorDataset(images[:40], labels[:40])
valid_ds = TensorDataset(images[40:], labels[40:])
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False)

model = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(4 * 8 * 8, 2)).to(device)
loss_fn = nn.CrossEntropyLoss(); optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

for _ in range(3):
    train_one_epoch(model, train_loader, loss_fn, optimizer)
xb, yb = next(iter(valid_loader)); xb = xb.to(device)

class_to_idx = {'vertical': 0, 'horizontal': 1}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

model.eval()
with torch.no_grad():
    logits = model(xb)
    probabilities = torch.softmax(logits, dim=1)
    row_sums = probabilities.sum(dim=1)
    assert torch.allclose(
        row_sums, torch.ones_like(row_sums), atol=1e-6
    )
    pred_indices = probabilities.argmax(dim=1)
    confidences = probabilities.gather(
        1, pred_indices.unsqueeze(1)
    ).squeeze(1)

prediction_results = [
    {'class_name': idx_to_class[idx], 'confidence': round(confidence, 4)}
    for idx, confidence in zip(
        pred_indices.cpu().tolist(), confidences.cpu().tolist()
    )
]
print(prediction_results)


### 해설 및 실행 결과 해석

- inference에서는 eval()과 no_grad()를 함께 사용합니다. softmax 행 합을 확인한 뒤 argmax index를 저장된 class mapping으로 되돌리면, CPU Python 리스트에 사람이 읽을 class 이름과 confidence를 함께 남길 수 있습니다.
